# [Chapter 11 — Model Generalizations, §11.3] The $SEIR_I$ Model: Adding a Latent Period

**Book:** *Essential Considerations for Modeling Epidemics* (Hyman/Qu/Xue, 2026)
**Chapter:** Chapter 11 — Model Generalizations
**Considerations developed:** 5 (linearization fidelity), 10 (generation-time-distribution misspecification)
**Estimated runtime:** ≤ 30 s

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/machyman/hyman2026essential/blob/main/python/notebooks/chapter_11_generalizations/01_SEIR_latent_period.ipynb)


## What this notebook does

Compares the basic $SIR_I$ model against an $SEIR_I$ model that adds an exposed compartment $E$ with mean latent period $1/\sigma$. Both have the same $\mathcal{R}_0$ and the same final attack rate, but the $SEIR_I$ epidemic peaks later and slightly lower. The notebook quantifies that delay and discusses why a model that ignores the latent period misestimates the generation time (Consideration~10) and consequently misestimates $\mathcal{R}_e(t)$ from incidence data.

## Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint
from shared import book_style, BOOK_COLORS
from shared.seeds import set_seed_chapter_11
from shared.verification import assert_within_tolerance
set_seed_chapter_11()
book_style()


## SIR vs SEIR with matched $\mathcal{R}_0$

We use the chapter-5 baseline ($\mathcal{R}_0 = 4$, mean infectious period $\tau_R = 10$ days). The SEIR variant uses $1/\sigma = 3$ days for the latent period.

In [ ]:
from shared.parameters import baseline_chapter_11
from shared.solvers import integrate_sir_i
p = baseline_chapter_11()

# SIR_I baseline
sir = integrate_sir_i(p)

# SEIR system: dS = -beta c_I S I/N; dE = +beta c_I S I/N - sigma E; dI = sigma E - I/tau_R; dR = I/tau_R
def seir_rhs(y, t, p):
    S, E, I, R = y
    N = p['N']
    incidence = p['c_I'] * p['beta'] * S * I / N
    return [-incidence,
            incidence - p['sigma']*E,
            p['sigma']*E - I/p['tau_R'],
            I/p['tau_R']]

y0 = [p['S0'], p['E0'] + 1e-6, p['I0'], p['R0_init']]
t = np.linspace(p['t_start'], p['t_end'], p['n_points'])
sol = odeint(seir_rhs, y0, t, args=(p,))
S_se, E_se, I_se, R_se = sol.T


### Visualizing the delay

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.plot(sir['t'], sir['I'], color=BOOK_COLORS['infectious'], lw=2, label='SIR$_I$')
ax.plot(t, I_se, color=BOOK_COLORS['secondary'], lw=2, ls='--', label='SEIR$_I$')
ax.set_xlabel('time (days)')
ax.set_ylabel('infectious fraction $I$')
ax.set_title('Latent period delays the epidemic peak (matched $R_0$)')
ax.legend()
fig.tight_layout()
plt.show()

peak_sir = sir['t'][int(np.argmax(sir['I']))]
peak_seir = t[int(np.argmax(I_se))]
print(f'SIR peak time:  {peak_sir:.1f} d')
print(f'SEIR peak time: {peak_seir:.1f} d')
print(f'Delay attributable to latent period: {peak_seir - peak_sir:.1f} d')


### Final attack rate is preserved (Consideration~5)

Linearization at the disease-free equilibrium gives the same dominant eigenvalue and same final-size equation for both models. The latent period changes the *shape* of the epidemic, not its eventual extent.

In [ ]:
final_sir = float(sir['R'][-1] + sir['I'][-1])
final_seir = float(R_se[-1] + I_se[-1] + E_se[-1])
print(f'Final attack rate, SIR  = {final_sir:.4f}')
print(f'Final attack rate, SEIR = {final_seir:.4f}')
assert_within_tolerance(final_sir, final_seir, abs_tol=0.01,
    label='Final attack rate matches between SIR and SEIR with same R_0')


## What's next

- Notebook 02 explores loss of immunity ($SIRS_I$), which qualitatively changes long-term behavior.
- Chapter 12 introduces population heterogeneity, the next step beyond compartmental refinement.
- The mismatch between the SIR and SEIR generation-time distributions, even with matched $\mathcal{R}_0$, biases $\mathcal{R}_e(t)$ estimates from incidence data — see book §10.3 for the worked-out distortion.